# Importing necessary Libraries

In [24]:
import pandas as pd

# Step 1: Data Understanding & Preprocessing

## Loading Data

In [25]:
df = pd.read_excel("./Data/REPORT.xlsx")

In [26]:
df

,YEAR,BARCODE,NAME,DIV,DEP,LOC,FE,STATUS,LEAVE DATE,TYPE,NO DAYS
0,2020.0,26.0,M.G.R.LIVERA,DYA,SMT,PP4,1.0,ACTIVE,2020-08-20,AL,1.0
1,2020.0,26.0,M.G.R.LIVERA,DYA,SMT,PP4,1.0,ACTIVE,2020-08-28,AL,1.0
2,2020.0,26.0,M.G.R.LIVERA,DYA,SMT,PP4,1.0,ACTIVE,2020-07-06,AL,0.5
3,2020.0,26.0,M.G.R.LIVERA,DYA,SMT,PP4,1.0,ACTIVE,2020-09-08,AL,1.0
4,2020.0,26.0,M.G.R.LIVERA,DYA,SMT,PP4,1.0,ACTIVE,2020-09-09,AL,1.0
...,...,...,...,...,...,...,...,...,...,...,...
257766,2025.0,5355.0,A.M.K.M.ARAMPOLA,DPR,SMO,MAO,2.0,ACTIVE,2025-02-24,AL,1.0
257767,2025.0,5355.0,A.M.K.M.ARAMPOLA,DPR,SMO,MAO,2.0,ACTIVE,2025-02-25,AL,1.0
257768,2025.0,5355.0,A.M.K.M.ARAMPOLA,DPR,SMO,MAO,2.0,ACTIVE,2025-02-26,AL,1.0
257769,2025.0,5356.0,P.D.C. PREMACHANDRA,DPR,SSO,CAR,2.0,ACTIVE,2025-02-24,AL,0.5


## Handle missing values

In [27]:
df.isnull().sum()

YEAR          14822
BARCODE       14822
NAME          14822
DIV           14822
DEP           14822
LOC           14822
FE            14822
STATUS        14822
LEAVE DATE    14822
TYPE          19311
NO DAYS       14822
dtype: int64

### Handling the missing values of the `LEAVE TYPE` column using the mode of each employee

In [28]:
# Drop rows where 'NAME' is NaN
df = df.dropna(subset=['NAME'])

In [29]:
df.isnull().sum()

YEAR             0
BARCODE          0
NAME             0
DIV              0
DEP              0
LOC              0
FE               0
STATUS           0
LEAVE DATE       0
TYPE          4489
NO DAYS          0
dtype: int64

In [30]:
# Compute the mode of leave_type for each employee
mode_per_employee = df.groupby('NAME')['TYPE'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)

# Fill null values in leave_type using the computed mode
df['TYPE'] = df.apply(lambda row: mode_per_employee[row['NAME']] if pd.isna(row['TYPE']) else row['TYPE'], axis=1)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_20552\893224361.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['TYPE'] = df.apply(lambda row: mode_per_employee[row['NAME']] if pd.isna(row['TYPE']) else row['TYPE'], axis=1)


In [31]:
df.isnull().sum()

YEAR          0
BARCODE       0
NAME          0
DIV           0
DEP           0
LOC           0
FE            0
STATUS        0
LEAVE DATE    0
TYPE          0
NO DAYS       0
dtype: int64

In [32]:
df.nunique()

YEAR             6
BARCODE       1563
NAME          1562
DIV             13
DEP             41
LOC             93
FE               4
STATUS           1
LEAVE DATE    1832
TYPE            17
NO DAYS          2
dtype: int64

In [33]:
df = df.drop(columns=['YEAR', 'FE', 'STATUS'])

## Feature Engineering

### Converting "LEAVE DATE" to datatime format

In [34]:
df['LEAVE DATE'] = pd.to_datetime(df['LEAVE DATE'], errors='coerce')

In [35]:
df.describe()

,BARCODE,LEAVE DATE,NO DAYS
count,242949.000000,242949,242949.000000
mean,3406.992682,2022-10-27 03:03:22.486941696,0.845636
min,26.000000,2019-01-30 00:00:00,0.500000
25%,2799.000000,2021-08-19 00:00:00,0.500000
50%,3363.000000,2022-11-05 00:00:00,1.000000
75%,4099.000000,2024-01-27 00:00:00,1.000000
max,5357.000000,2025-06-04 00:00:00,1.000000
std,962.564840,NaN,0.230985


In [ ]:
# Filter out rows with LEAVE YEAR as 2025
df = df[df['LEAVE DATE'].dt.year != 2025]


### Extract Month, Day of the Week, and Year

In [10]:
df['LEAVE MONTH'] = df['LEAVE DATE'].dt.month
df['LEAVE YEAR'] = df['LEAVE DATE'].dt.year
df['LEAVE DAY OF WEEK'] = df['LEAVE DATE'].dt.dayofweek

```python
df['LEAVE DATE'].dt.month
```
- Extracts the month from the LEAVE DATE column.
- Returns the month as a number (e.g., 1 for January, 2 for February, etc.). 

```python
df['LEAVE DATE'].dt.dayofweek
```
- Extracts the day of the week from the LEAVE DATE column.
- Returns the day of the week as a number, where:
    - 0 = Monday
    - 1 = Tuesday
    - 2 = Wednesday
    - 3 = Thursday
    - 4 = Friday
    - 5 = Saturday
    - 6 = Sunday

In [11]:
df

,SERVICE NO,DIV,DEP,LOC,LEAVE DATE,LEAVE TYPE,NO DAYS,LEAVE MONTH,LEAVE YEAR,LEAVE DAY OF WEEK
0,M.G.R.LIVERA,DYA,SMT,PP4,2024-01-03,AL,1.0,1,2024,2
1,M.G.R.LIVERA,DYA,SMT,PP4,2024-01-06,AL,0.5,1,2024,5
2,M.G.R.LIVERA,DYA,SMT,PP4,2024-05-20,AL,1.0,5,2024,0
3,M.G.R.LIVERA,DYA,SMT,PP4,2024-04-03,AL,1.0,4,2024,2
4,M.G.R.LIVERA,DYA,SMT,PP4,2024-06-15,AL,0.5,6,2024,5
...,...,...,...,...,...,...,...,...,...,...
57635,K.P.A.DULANJANA,DPR,SDE,MAS,2024-11-26,SK,1.0,11,2024,1
57636,H.M.Y.G.V.S. SENAVIRATHNA,DYA,SMT,MAT,2024-11-14,DL,1.0,11,2024,3
57637,R.A.S.DHARMAWANSHA,DPR,SEA,ELW,2024-12-28,SK,0.5,12,2024,5
57638,D.M.P.MADHUSANKA,DMD,SRC,FIU,2024-12-30,SK,0.5,12,2024,0


### Calculate Time Gap Between Leaves

In [12]:
# Sort data by employee and leave date
df = df.sort_values(by=['SERVICE NO', 'LEAVE DATE'])

#### Calculating days since last leave

In [13]:
# Compute difference between consecutive leaves per employee
df['PREV LEAVE DATE'] = df.groupby('SERVICE NO')['LEAVE DATE'].shift(1)
df['DAYS SINCE LAST LEAVE'] = (df['LEAVE DATE'] - df['PREV LEAVE DATE']).dt.days

#### Calculating days until next leave

In [14]:
df['NEXT LEAVE DATE'] = df.groupby('SERVICE NO')['LEAVE DATE'].shift(-1)
df['DAYS UNTIL NEXT LEAVE'] = (df['NEXT LEAVE DATE'] - df['LEAVE DATE']).dt.days

In [15]:
df

,SERVICE NO,DIV,DEP,LOC,LEAVE DATE,LEAVE TYPE,NO DAYS,LEAVE MONTH,LEAVE YEAR,LEAVE DAY OF WEEK,PREV LEAVE DATE,DAYS SINCE LAST LEAVE,NEXT LEAVE DATE,DAYS UNTIL NEXT LEAVE
9658,A.A.A.R.DE SILVA,DHA,SAB,SEU,2024-01-04,CS,1.0,1,2024,3,NaT,NaN,2024-01-24,20.0
9640,A.A.A.R.DE SILVA,DHA,SAB,SEU,2024-01-24,AL,1.0,1,2024,2,2024-01-04,20.0,2024-01-26,2.0
9641,A.A.A.R.DE SILVA,DHA,SAB,SEU,2024-01-26,AL,1.0,1,2024,4,2024-01-24,2.0,2024-01-27,1.0
9642,A.A.A.R.DE SILVA,DHA,SAB,SEU,2024-01-27,AL,0.5,1,2024,5,2024-01-26,1.0,2024-02-03,7.0
9654,A.A.A.R.DE SILVA,DHA,SAB,SEU,2024-02-03,CS,0.5,2,2024,5,2024-01-27,7.0,2024-03-09,35.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56606,Y.S.D.DHARMATHILAKA,DMD,SRC,FIU,2024-09-21,DL,0.5,9,2024,5,2024-07-23,60.0,2024-10-02,11.0
56605,Y.S.D.DHARMATHILAKA,DMD,SRC,FIU,2024-10-02,CS,1.0,10,2024,2,2024-09-21,11.0,2024-10-07,5.0
56609,Y.S.D.DHARMATHILAKA,DMD,SRC,FIU,2024-10-07,SK,1.0,10,2024,0,2024-10-02,5.0,2024-10-08,1.0
56610,Y.S.D.DHARMATHILAKA,DMD,SRC,FIU,2024-10-08,SK,1.0,10,2024,1,2024-10-07,1.0,2024-11-14,37.0


#### Calculating the cumulative leave count for each year

In [16]:
# Sort by employee and leave_date
df = df.sort_values(by=['SERVICE NO', 'LEAVE YEAR', 'LEAVE DATE'])

# Compute cumulative leave count per year for each employee
df['CUMULATIVE LEAVE COUNT'] = df.groupby(['SERVICE NO', 'LEAVE YEAR']).cumcount() + 1


- `CUMULATIVE LEAVE COUNT` resets every new year for each employee.

In [17]:
# Print an example for an employee to verify the answer
df[df['SERVICE NO'] == "A.A.A.R.DE SILVA"]

,SERVICE NO,DIV,DEP,LOC,LEAVE DATE,LEAVE TYPE,NO DAYS,LEAVE MONTH,LEAVE YEAR,LEAVE DAY OF WEEK,PREV LEAVE DATE,DAYS SINCE LAST LEAVE,NEXT LEAVE DATE,DAYS UNTIL NEXT LEAVE,CUMULATIVE LEAVE COUNT
9658,A.A.A.R.DE SILVA,DHA,SAB,SEU,2024-01-04,CS,1.0,1,2024,3,NaT,NaN,2024-01-24,20.0,1
9640,A.A.A.R.DE SILVA,DHA,SAB,SEU,2024-01-24,AL,1.0,1,2024,2,2024-01-04,20.0,2024-01-26,2.0,2
9641,A.A.A.R.DE SILVA,DHA,SAB,SEU,2024-01-26,AL,1.0,1,2024,4,2024-01-24,2.0,2024-01-27,1.0,3
9642,A.A.A.R.DE SILVA,DHA,SAB,SEU,2024-01-27,AL,0.5,1,2024,5,2024-01-26,1.0,2024-02-03,7.0,4
9654,A.A.A.R.DE SILVA,DHA,SAB,SEU,2024-02-03,CS,0.5,2,2024,5,2024-01-27,7.0,2024-03-09,35.0,5
9645,A.A.A.R.DE SILVA,DHA,SAB,SEU,2024-03-09,AL,0.5,3,2024,5,2024-02-03,35.0,2024-03-21,12.0,6
9653,A.A.A.R.DE SILVA,DHA,SAB,SEU,2024-03-21,CS,1.0,3,2024,3,2024-03-09,12.0,2024-04-20,30.0,7
9652,A.A.A.R.DE SILVA,DHA,SAB,SEU,2024-04-20,AL,0.5,4,2024,5,2024-03-21,30.0,2024-04-24,4.0,8
9655,A.A.A.R.DE SILVA,DHA,SAB,SEU,2024-04-24,CS,1.0,4,2024,2,2024-04-20,4.0,2024-04-25,1.0,9
9656,A.A.A.R.DE SILVA,DHA,SAB,SEU,2024-04-25,CS,1.0,4,2024,3,2024-04-24,1.0,2024-05-08,13.0,10


Getting unique values of the `LEAVE TYPE` column

In [18]:
unique_values = df['LEAVE TYPE'].unique()
unique_values

array(['CS', 'AL', 'SK', 'DL', 'SP', 'AC', 'SD', 'SU'], dtype=object)

### Calculates the leave type distribution per employee

In [19]:
# # Get total leaves per employee
# total_leaves_per_employee = df.groupby('SERVICE NO')['LEAVE TYPE'].count()
# 
# # Get leave type counts per employee
# leave_type_distribution = df.groupby(['SERVICE NO', 'LEAVE TYPE']).size().unstack(fill_value=0)
# 
# # Convert counts to proportions
# leave_type_distribution = leave_type_distribution.div(total_leaves_per_employee, axis=0)
# 
# # Merge with main dataframe
# df = df.merge(leave_type_distribution, on='SERVICE NO', how='left')
# df

In [20]:
df.to_csv('./Data/after_preprocess.csv', index=False)